In [1]:
# importing libraries
import pandas as pd
import numpy as np
import joblib

import torch
import torch.nn as nn

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

In [2]:
# loading dataset
df = pd.read_csv("train_svm_extended_all.csv")

print("Shape:", df.shape)

print("\nClass Distribution:")
print(df["label"].value_counts())

df.head()

Shape: (4527, 4)

Class Distribution:
label
1    2287
0    2240
Name: count, dtype: int64


,quality_score,best_similarity,margin,label
0,27.075550,0.574058,0.413278,1
1,15.175558,0.443810,0.227276,1
2,20.051987,0.505812,0.315841,1
3,16.646429,0.373729,0.075268,1
4,19.913544,0.537300,0.352941,1


In [3]:
# features and labels
X = df[["quality_score", "best_similarity", "margin"]]

y = df["label"]

print(X.shape)
print(y.shape)

(4527, 3)
(4527,)


In [4]:
# training validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)

Train: (3621, 3)
Validation: (906, 3)


In [5]:
# feature scaling
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)

joblib.dump(scaler, "mlp_scaler.pkl")

print("Scaler Saved")

Scaler Saved


In [6]:
# converting to torch sensors
X_train_tensor = torch.tensor(X_train_scaled, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.long)

X_val_tensor = torch.tensor(X_val_scaled, dtype=torch.float32)

y_val_tensor = torch.tensor(y_val.values, dtype=torch.long)

print(X_train_tensor.shape)
print(y_train_tensor.shape)

torch.Size([3621, 3])
torch.Size([3621])


In [7]:
# defining simple one layer MLP
class SimpleMLP(nn.Module):

    def __init__(self):

        super().__init__()

        self.network = nn.Sequential(nn.Linear(3, 8), nn.ReLU(), nn.Linear(8, 2))

    def forward(self, x):

        return self.network(x)

In [8]:
# creating model
model = SimpleMLP()

print(model)

SimpleMLP(
  (network): Sequential(
    (0): Linear(in_features=3, out_features=8, bias=True)
    (1): ReLU()
    (2): Linear(in_features=8, out_features=2, bias=True)
  )
)


In [9]:
# loss and optimizer
criterion = nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

In [10]:
# training loop
EPOCHS = 100

best_val_acc = 0

for epoch in range(EPOCHS):

    model.train()

    optimizer.zero_grad()

    outputs = model(X_train_tensor)

    loss = criterion(outputs, y_train_tensor)

    loss.backward()

    optimizer.step()

    model.eval()

    with torch.no_grad():

        val_outputs = model(X_val_tensor)

        val_predictions = torch.argmax(val_outputs, dim=1)

        val_acc = accuracy_score(y_val_tensor.numpy(), val_predictions.numpy())

    if val_acc > best_val_acc:

        best_val_acc = val_acc

        torch.save(model.state_dict(), "mlp_model.pth")

    if epoch % 10 == 0:

        print(
            f"Epoch {epoch:3d} | "
            f"Loss: {loss.item():.4f} | "
            f"Val Acc: {val_acc:.4f}"
        )

print("\nBest Validation Accuracy:", best_val_acc)

Epoch   0 | Loss: 0.5947 | Val Acc: 0.7737
Epoch  10 | Loss: 0.5632 | Val Acc: 0.8256
Epoch  20 | Loss: 0.5326 | Val Acc: 0.8819
Epoch  30 | Loss: 0.5029 | Val Acc: 0.9283
Epoch  40 | Loss: 0.4741 | Val Acc: 0.9503
Epoch  50 | Loss: 0.4462 | Val Acc: 0.9603
Epoch  60 | Loss: 0.4192 | Val Acc: 0.9625
Epoch  70 | Loss: 0.3929 | Val Acc: 0.9691
Epoch  80 | Loss: 0.3677 | Val Acc: 0.9735
Epoch  90 | Loss: 0.3435 | Val Acc: 0.9757

Best Validation Accuracy: 0.9768211920529801


In [11]:
# load best model
best_model = SimpleMLP()

best_model.load_state_dict(torch.load("mlp_model.pth"))

best_model.eval()

print("Best Model Loaded")

Best Model Loaded


In [12]:
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
    classification_report,
)

with torch.no_grad():

    outputs = best_model(X_val_tensor)

    predictions = torch.argmax(outputs, dim=1)

y_true = y_val_tensor.numpy()

y_pred = predictions.numpy()

accuracy = accuracy_score(y_true, y_pred)

precision = precision_score(y_true, y_pred)

recall = recall_score(y_true, y_pred)

f1 = f1_score(y_true, y_pred)

print("Accuracy :", accuracy)
print("Precision:", precision)
print("Recall   :", recall)
print("F1 Score :", f1)

print("\nConfusion Matrix")
print(confusion_matrix(y_true, y_pred))

print("\nClassification Report")
print(classification_report(y_true, y_pred))

Accuracy : 0.9768211920529801
Precision: 0.9888143176733781
Recall   : 0.9650655021834061
F1 Score : 0.9767955801104973

Confusion Matrix
[[443   5]
 [ 16 442]]

Classification Report
              precision    recall  f1-score   support

           0       0.97      0.99      0.98       448
           1       0.99      0.97      0.98       458

    accuracy                           0.98       906
   macro avg       0.98      0.98      0.98       906
weighted avg       0.98      0.98      0.98       906

